# Experiment: FK LSTM Colab Batch Runner

Objective:
- Run the Fischer-Krauss LSTM batch suite on Google Colab with GPU support.
- Save artifacts to Google Drive so interrupted sessions can resume.
- Start with the lighter `configs/fk_lstm_fast/` presets, then switch to `configs/fk_lstm/` for the full suite.


## Plan

- Mount Drive and choose where the repo and artifacts should live.
- Install the minimum runtime dependencies needed for FK training and dashboard rendering.
- Run the batch runner with `--output-root` and `--skip-existing` so repeated Colab sessions can resume.
- Inspect the generated comparison CSV and HTML index directly from the notebook.


In [ ]:
# Colab setup: mount Drive and clone the repo once.
from __future__ import annotations

import os
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except ImportError:
    print('Not running inside Colab. Adjust paths below for local notebook execution.')

REPO_URL = 'https://github.com/<your-org>/data_stock_market.git'
WORK_ROOT = Path('/content/drive/MyDrive/fk_lstm_runs')
REPO_DIR = WORK_ROOT / 'data_stock_market'
ARTIFACT_ROOT = WORK_ROOT / 'artifacts'
INDEX_OUTPUT = ARTIFACT_ROOT / 'run_index' / 'fk_fast_dashboard.html'
CONFIG_DIR = 'configs/fk_lstm_fast'

WORK_ROOT.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Reusing existing repo at {REPO_DIR}')

%cd {REPO_DIR}
print('Repo:', REPO_DIR)
print('Artifacts:', ARTIFACT_ROOT)


In [ ]:
# Runtime setup. Colab usually has TensorFlow already, but we pin the missing pieces here.
%pip install -q pandas matplotlib yfinance beautifulsoup4 pydantic packaging ipython

import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))


In [ ]:
# Choose fast vs full research suite.
# For a first pass, keep CONFIG_DIR = 'configs/fk_lstm_fast'.
# When that looks good, switch to 'configs/fk_lstm' for the full walk-forward suite.
CONFIG_DIR = 'configs/fk_lstm_fast'
EXTRA_ARGS = ''

print('Config dir:', CONFIG_DIR)
print('Extra args:', EXTRA_ARGS)


In [ ]:
# Run the FK batch suite. --skip-existing makes the notebook resumable after Colab disconnects.
cmd = (
    'PYTHONPATH=src/training python3 src/training/scripts/run_fk_batch.py '
    f'--config-dir {CONFIG_DIR} '
    f'--output-root {ARTIFACT_ROOT / "fk_lstm_classifier"} '
    f'--index-output {INDEX_OUTPUT} '
    '--skip-existing '
    f'{EXTRA_ARGS}'
)
print(cmd)
!{cmd}


In [ ]:
# Review the batch summary table.
import pandas as pd

summary_path = INDEX_OUTPUT.with_suffix('.csv')
summary_df = pd.read_csv(summary_path)
summary_df.sort_values(['auc_rank', 'final_equity'], ascending=False).reset_index(drop=True)


In [ ]:
# Open the generated HTML comparison index and one run folder.
from IPython.display import FileLink, display

display(FileLink(str(INDEX_OUTPUT)))
best_run_dir = Path(summary_df.sort_values('final_equity', ascending=False).iloc[0]['run_dir'])
display(FileLink(str(best_run_dir / 'dashboard.html')))
best_run_dir


## Results

- Use the CSV table for quick ranking across `VN` and `US` fast configs.
- Use the HTML dashboards only for the top candidates; the batch index already compresses the main comparison.
- If Colab disconnects, rerun the batch cell. Completed runs will be skipped automatically when `--skip-existing` is enabled.


## Next steps

- After the fast suite stabilizes, switch `CONFIG_DIR` to `configs/fk_lstm` for the full research batch.
- If one market dominates runtime, run `VN` and `US` in separate notebooks so failures do not block the other market.
- Promote only the final comparison CSV and selected dashboards; keep raw Colab scratch outputs under Drive.
